# Packet P-047 — Kitchen Sink Panel Revalidation

**Decision question:** P-046 found +0.040 tau-b lift with 31 features (adding solvents, layer thicknesses, thermal exposure, LLE embeddings). Does the expanded model change the panel rankings? Are the 3 locked panel devices still top-ranked, or do better candidates emerge?

**Fastest falsifier:** Run 20-split ranking with 31-feature model. If panel devices drop below top-20, the expanded model changes device selection.

**Success:** Panel devices remain top-20 in ≥90% of splits with expanded features.
**Kill:** Any panel device drops below top-50% — expanded model invalidates current panel.

In [1]:
"""Cell 1 — Panel ranking with 31-feature kitchen sink model."""
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split
from scipy.stats import kendalltau
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('perovskite_stability_clean.csv')

ORIGINAL_16 = [
    'Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
    'MA', 'FA', 'Cs',
    'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
    'Perovskite_thickness', 'Cell_area_measured',
    'JV_default_Voc', 'JV_default_Jsc', 'JV_default_FF'
]

KITCHEN_SINK = ORIGINAL_16 + [
    'DMF', 'DMSO', 'other_solvent', 'DMF_DMSO_ratio',
    'LLE_1', 'LLE_2', 'LLE_3', 'LLE_4',
    'Perovskite_annealing_thermal_exposure',
    'Backcontact_thickness_list', 'ETL_thickness', 'HTL_thickness_list',
    'others_A', 'others_B', 'others_X'
]

TARGET = 'Stability_PCE_T80'
ET_PARAMS = dict(n_estimators=700, max_features='sqrt', min_samples_split=3,
                 min_samples_leaf=1, bootstrap=False, random_state=42)

PANEL = {850: 'MA₃Pb₄I₁₃', 1320: 'MA₀.₂₅FA₀.₇₅PbI₂.₇₇Br₀.₂₅', 119: 'FA₀.₈₃Cs₀.₁₇PbI₂Br'}

N_SPLITS = 20

# Compare original 16 vs kitchen sink 31
for model_name, feats in [('Original 16', ORIGINAL_16), ('Kitchen Sink 31', KITCHEN_SINK)]:
    X = df[feats].fillna(0)
    y = np.log1p(df[TARGET])
    
    panel_stats = {d: {'appearances': 0, 'top20': 0, 'top10': 0, 'ranks': [], 'percentiles': []} for d in PANEL}
    all_taus = []
    
    for seed in range(N_SPLITS):
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed)
        
        model = ExtraTreesRegressor(**ET_PARAMS)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_te)
        
        tau, _ = kendalltau(y_te, preds)
        all_taus.append(tau)
        
        test_idx = X_te.index.tolist()
        ranked = sorted(zip(test_idx, preds), key=lambda x: -x[1])
        rank_map = {idx: rank for rank, (idx, _) in enumerate(ranked)}
        n_test = len(ranked)
        
        for d in PANEL:
            if d not in rank_map:
                continue
            panel_stats[d]['appearances'] += 1
            rank = rank_map[d]
            panel_stats[d]['ranks'].append(rank)
            panel_stats[d]['percentiles'].append(1 - rank / n_test)
            if rank < 20:
                panel_stats[d]['top20'] += 1
            if rank < 10:
                panel_stats[d]['top10'] += 1
    
    print(f"\n{'=' * 75}")
    print(f"PANEL RANKING — {model_name}")
    print(f"Mean tau-b: {np.mean(all_taus):.4f}")
    print(f"{'=' * 75}")
    print(f"{'Device':<10} {'Name':<30} {'Apps':>5} {'Top-20':>8} {'Top-10':>8} {'Mean rank':>10} {'Pctile':>8}")
    print("-" * 82)
    
    for d, name in PANEL.items():
        s = panel_stats[d]
        apps = s['appearances']
        if apps == 0:
            print(f"{d:<10} {name:<30} {'N/A':>5}")
            continue
        t20_rate = s['top20'] / apps
        t10_rate = s['top10'] / apps
        mean_rank = np.mean(s['ranks'])
        mean_pct = np.mean(s['percentiles'])
        print(f"{d:<10} {name:<30} {apps:>5} {t20_rate:>7.0%} {t10_rate:>7.0%} {mean_rank:>10.1f} {mean_pct:>7.1%}")

# === Check if new top devices emerge ===
print(f"\n{'=' * 75}")
print("TOP-20 FREQUENCY — KITCHEN SINK MODEL")
print(f"{'=' * 75}")

X_ks = df[KITCHEN_SINK].fillna(0)
y = np.log1p(df[TARGET])

device_top20_count = {}
for seed in range(N_SPLITS):
    X_tr, X_te, y_tr, y_te = train_test_split(X_ks, y, test_size=0.2, random_state=seed)
    model = ExtraTreesRegressor(**ET_PARAMS)
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    
    test_idx = X_te.index.tolist()
    ranked = sorted(zip(test_idx, preds), key=lambda x: -x[1])
    top20_idx = [idx for idx, _ in ranked[:20]]
    
    for idx in top20_idx:
        device_top20_count[idx] = device_top20_count.get(idx, 0) + 1

# Show top candidates by frequency
freq_df = pd.DataFrame([
    {'device': idx, 'top20_count': count, 'top20_rate': count / N_SPLITS,
     'actual_T80': df.loc[idx, TARGET],
     'composition': str(df.loc[idx, 'Perovskite_composition_long_form'])[:50],
     'is_panel': idx in PANEL}
    for idx, count in device_top20_count.items()
]).sort_values('top20_count', ascending=False)

print(f"\n{'Device':<8} {'Top-20 rate':>12} {'T80 (h)':>10} {'Panel?':>8} {'Composition':<50}")
print("-" * 92)
for _, row in freq_df.head(20).iterrows():
    marker = "  <<<" if row['is_panel'] else ""
    print(f"{int(row['device']):<8} {row['top20_rate']:>11.0%} {row['actual_T80']:>10.0f} {str(row['is_panel']):>8} {row['composition']:<50}{marker}")


PANEL RANKING — Original 16
Mean tau-b: 0.2999
Device     Name                            Apps   Top-20   Top-10  Mean rank   Pctile
----------------------------------------------------------------------------------
850        MA₃Pb₄I₁₃                          4    100%    100%        0.0  100.0%
1320       MA₀.₂₅FA₀.₇₅PbI₂.₇₇Br₀.₂₅          3    100%     67%        6.7   97.8%
119        FA₀.₈₃Cs₀.₁₇PbI₂Br                 4    100%     75%        8.5   97.2%



PANEL RANKING — Kitchen Sink 31
Mean tau-b: 0.3389
Device     Name                            Apps   Top-20   Top-10  Mean rank   Pctile
----------------------------------------------------------------------------------
850        MA₃Pb₄I₁₃                          4    100%    100%        0.2   99.9%
1320       MA₀.₂₅FA₀.₇₅PbI₂.₇₇Br₀.₂₅          3    100%     67%        8.0   97.4%
119        FA₀.₈₃Cs₀.₁₇PbI₂Br                 4    100%     25%       10.5   96.6%

TOP-20 FREQUENCY — KITCHEN SINK MODEL



Device    Top-20 rate    T80 (h)   Panel? Composition                                       
--------------------------------------------------------------------------------------------
677              40%         61    False MAPbI3                                            
1380             35%       1824    False MAPbI3                                            
267              35%        600    False FA0.6MA0.4PbI3                                    
966              35%        100    False (NH4)6.8FA0.15MA1.7Pb7.8Br0.45I23.8               
489              30%        800    False MAPbI3                                            
1373             30%       5000    False MAPbI3                                            
289              25%       2300    False MAPbI3                                            
1066             25%       5400    False FA0.85MA0.15PbBr0.45I2.55                         
975              25%       1008    False MAPbI3                              

In [2]:
"""Cell 2 — Evaluate and save."""

# Check panel status with kitchen sink model
panel_pass = True
for d in PANEL:
    s = panel_stats[d]
    if s['appearances'] > 0 and s['top20'] / s['appearances'] < 0.90:
        panel_pass = False

if panel_pass:
    status = "Confirmed"
else:
    any_bottom = False
    for d in PANEL:
        s = panel_stats[d]
        if s['appearances'] > 0 and np.mean(s['percentiles']) < 0.50:
            any_bottom = True
    status = "Negative" if any_bottom else "Promising"

freq_df.to_csv('Packet_P047_Kitchen_Sink_Panel.csv', index=False)
print(f"Saved: Packet_P047_Kitchen_Sink_Panel.csv")

print(f"\n{'=' * 60}")
print(f"P-047 STATUS: {status}")
print(f"{'=' * 60}")
if status == "Confirmed":
    print("Panel devices survive the kitchen sink model upgrade.")
    print("The +0.040 tau-b lift doesn't change device selection.")
elif status == "Negative":
    print("Panel devices are NOT well-ranked with expanded features.")
    print("Kitchen sink model may identify better candidates.")
else:
    print("Mixed results — some panel devices are borderline.")

Saved: Packet_P047_Kitchen_Sink_Panel.csv

P-047 STATUS: Confirmed
Panel devices survive the kitchen sink model upgrade.
The +0.040 tau-b lift doesn't change device selection.
